<a href="https://colab.research.google.com/github/blue0454/OSP-TEAM37/blob/main/kobert_token_classification_allergen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate accelerate
!pip install seqeval  # NER(개체명 인식) 모델의 F1-score 성능 평가를 위한 필수 라이브러리

In [ ]:
#구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# 'kobert_ner_sample.csv' 파일의 실제 Google Drive 경로로 이 부분을 교체하세요.
# 예시: '/content/drive/MyDrive/kobert_ner_sample.csv' 또는
#       '/content/drive/MyDrive/MyFolder/kobert_ner_sample.csv'
file_path = '/content/drive/MyDrive/kobert_ner_sample.csv' # 여기에 정확한 파일 경로를 입력하세요.

try:
    df = pd.read_csv(file_path, encoding='cp949')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='euc-kr')
except FileNotFoundError:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요: {file_path}")
    print("Google Drive에 파일이 올바르게 위치해 있는지, 그리고 경로가 정확한지 확인해주세요.")
else:
    print("데이터 크기:", df.shape)
    display(df.head())

데이터 크기: (30, 6)


,image_name,ocr_text,corrected_text,tokens,labels,Unnamed: 5
0,test_20260614_134101_jpg.rf.31291d284c56619b56...,제품명 고구마형과자 내용량 162 @ 식품유형| 과자(유탕처리제품) (업소명 (주)...,인식 불가,NaN,NaN,unreadable
1,test_20260614_134258_jpg.rf.b5b5e0a48f4851716a...,린0 말리 기타 댁린 힘유 @로움점일행 한천 '힘유제 조건제어모제베진로생승 %) (...,인식 불가,NaN,NaN,unreadable
2,test_20260614_134750_jpg.rf.e796a2b61a281428b4...,50 5 제1 (국색스 Ne 밀 우유 대륙 [소비기화 |말런 또기히하집물표보고a ...,"밀, 우유, 대두, 아황산류 함유",밀|우유|대두|아황산류|함유,B-WHEAT|B-MILK|B-SOY|O|O,readable
3,test_20260614_134912_jpg.rf.fa9ad12d1b6eec7758...,용세의 35버i생인 Gni 농빛 Orone 용 LOTT 1952 E내a내교레교 ES...,"밀, 대두, 우유, 땅콩, 달걀, 쇠고기 함유",밀|대두|우유|땅콩|달걀|쇠고기|함유,B-WHEAT|B-SOY|B-MILK|B-PEANUT|B-EGG|O|O,readable
4,test_20260614_135034_jpg.rf.fce49c5fd42aff4cf8...,00 8 3 '어쨌리? 눈말 40 셋 233 (행점 '의한 유 힘 8 '7후리로L...,"우유, 밀, 대두 함유",우유|밀|대두|함유,B-MILK|B-WHEAT|B-SOY|O,readable


In [ ]:
import pandas as pd

# 1. 파일 불러오기 (인코딩 에러 방지 처리)
# Google Drive에서 파일을 로드하려면 절대 경로를 사용해야 합니다.
file_path = '/content/drive/MyDrive/kobert_ner_sample.csv' # 이 경로를 사용하도록 수정합니다.

try:
    df = pd.read_csv(file_path, encoding="cp949")
except Exception:
    df = pd.read_csv(file_path, encoding="euc-kr")

# 2. '인식 가능'한 정상 데이터만 필터링하기
# (중요: 수민 님이 학습시킬 코드는 문자가 깨진 '인식 불가' 데이터는 제외해야 합니다!)
df_clean = df[df['Unnamed: 5'] == 'readable'].dropna(subset=['tokens', 'labels'])

# 3. 데이터 개수와 상위 3개 샘플 확인하기
print(f"전체 데이터 개수: {len(df)}개")
print(f"학습 가능한 정상 데이터 개수: {len(df_clean)}개\n")

# 필요한 컬럼만 쏙 골라서 출력해보기
df_clean[['corrected_text', 'tokens', 'labels']].head(3)

전체 데이터 개수: 30개
학습 가능한 정상 데이터 개수: 25개



,corrected_text,tokens,labels
2,"밀, 우유, 대두, 아황산류 함유",밀|우유|대두|아황산류|함유,B-WHEAT|B-MILK|B-SOY|O|O
3,"밀, 대두, 우유, 땅콩, 달걀, 쇠고기 함유",밀|대두|우유|땅콩|달걀|쇠고기|함유,B-WHEAT|B-SOY|B-MILK|B-PEANUT|B-EGG|O|O
4,"우유, 밀, 대두 함유",우유|밀|대두|함유,B-MILK|B-WHEAT|B-SOY|O


In [ ]:
# 1. 파이프(|) 기호로 뭉쳐있는 텍스트를 파이썬 리스트로 쪼개기
df_clean['token_list'] = df_clean['tokens'].apply(lambda x: x.split('|'))
df_clean['label_list'] = df_clean['labels'].apply(lambda x: x.split('|'))

# 2. 우리 데이터셋에 존재하는 모든 고유 라벨(BIO 태그) 추출하기
unique_labels = set()
for labels in df_clean['label_list']:
    unique_labels.update(labels)

# 3. 정렬된 고유 라벨 리스트 생성 및 확인
unique_labels = sorted(list(unique_labels))
print("데이터에서 추출한 고유 라벨 목록:", unique_labels)

# 4. 고유 라벨 개수 확인 (이 개수만큼 KoBERT 최종 레이어 노드가 설정됩니다)
num_labels = len(unique_labels)
print(f"총 라벨 개수: {num_labels}개")

# 변환된 데이터 샘플 1개 확인해보기
print("\n[변환 완료 샘플 확인]")
print("Tokens 리스트:", df_clean['token_list'].iloc[0])
print("Labels 리스트:", df_clean['label_list'].iloc[0])

데이터에서 추출한 고유 라벨 목록: ['B-EGG', 'B-MILK', 'B-PEANUT', 'B-SOY', 'B-WHEAT', 'O']
총 라벨 개수: 6개

[변환 완료 샘플 확인]
Tokens 리스트: ['밀', '우유', '대두', '아황산류', '함유']
Labels 리스트: ['B-WHEAT', 'B-MILK', 'B-SOY', 'O', 'O']


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# 1. 컴퓨터가 인식할 수 있도록 라벨(문자) <-> ID(숫자) 변환 사전 만들기
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Label to ID 매핑:", label2id)
print("ID to Label 매핑:", id2label)

# 2. 누구나 무료로 접근 가능한 대한민국 표준 'KLUE-BERT' 모델로 변경
# 한국어 자연어 처리 평가 표준(KLUE) 모델이라 성능과 안정성이 보장됩니다.
model_checkpoint = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 3. 토큰 분류(Token Classification)용 프리트레인 모델 로드
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True # 라벨 개수(6개)를 모델에 맞추어 주입하는 옵션
)

print("\n[성공] KLUE-BERT 토크나이저 및 프리트레인 AI 모델 로드 완료!")

Label to ID 매핑: {'B-EGG': 0, 'B-MILK': 1, 'B-PEANUT': 2, 'B-SOY': 3, 'B-WHEAT': 4, 'O': 5}
ID to Label 매핑: {0: 'B-EGG', 1: 'B-MILK', 2: 'B-PEANUT', 3: 'B-SOY', 4: 'B-WHEAT', 5: 'O'}


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly in


[성공] KLUE-BERT 토크나이저 및 프리트레인 AI 모델 로드 완료!


In [ ]:
import torch
from datasets import Dataset

# 1. 서브 토큰화에 맞춰 라벨을 복사하고 정렬하는 핵심 함수 정의
def align_labels_with_tokens(token_list, label_list):
    # 문장을 KLUE-BERT 토크나이저 기준의 input_ids로 변환
    tokenized_inputs = tokenizer(
        token_list,
        is_split_into_words=True,
        truncation=True,
        padding=False
    )

    labels = []
    word_ids = tokenized_inputs.word_ids() # 각 토큰이 원래 어떤 단어였는지 매핑된 ID 추출

    for word_idx in word_ids:
        if word_idx is None:
            # [CLS], [SEP] 같은 특수 토큰은 학습에서 제외하도록 -100 지정 (PyTorch 표준)
            labels.append(-100)
        else:
            # 쪼개진 서브 토큰에 원래 단어의 라벨 문자열을 매칭하고, 이를 숫자 ID로 변환
            label_str = label_list[word_idx]
            labels.append(label2id[label_str])

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 2. 전체 clean 데이터셋을 순회하며 변환 데이터 적재
input_ids_list = []
attention_mask_list = []
labels_list = []

for _, row in df_clean.iterrows():
    result = align_labels_with_tokens(row['token_list'], row['label_list'])
    input_ids_list.append(result['input_ids'])
    attention_mask_list.append(result['attention_mask'])
    labels_list.append(result['labels'])

# 3. 허깅페이스 전용 Dataset 구조로 래핑 (학습기인 Trainer에 넣기 위함)
train_dataset = Dataset.from_dict({
    'input_ids': input_ids_list,
    'attention_mask': attention_mask_list,
    'labels': labels_list
})

print(f"학습용 허깅페이스 데이터셋 구축 완료!")
print(train_dataset)

# 데이터가 내부적으로 어떻게 바뀌었는지 샘플 1개 확인
print("\n[첫 번째 데이터 인코딩 결과 샘플]")
print("Input IDs (숫자로 변환된 단어들):", train_dataset[0]['input_ids'])
print("Labels (숫자로 변환된 정답 라벨):", train_dataset[0]['labels'])

학습용 허깅페이스 데이터셋 구축 완료!
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 25
})

[첫 번째 데이터 인코딩 결과 샘플]
Input IDs (숫자로 변환된 단어들): [2, 1111, 7282, 11667, 1376, 2334, 2066, 2397, 8158, 3]
Labels (숫자로 변환된 정답 라벨): [-100, 4, 1, 3, 5, 5, 5, 5, 5, -100]


In [ ]:
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. 문장들의 길이를 알아서 맞춰주는 패딩 도구 정의
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. 모델의 성능을 측정할 평가지표(F1-score) 계산 함수 정의
# 계획서에 명시한 'F1-score 검증'을 수행하기 위해 seqeval 라이브러리를 활용합니다.
metric = evaluate.load("seqeval")
label_list = unique_labels

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=-1)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

In [ ]:
import os

# 1. 저장할 폴더 만들기
output_dir = "./final_kobert_allergen_model"
os.makedirs(output_dir, exist_ok=True)

# 2. 만약 trainer 변수가 살아있다면 trainer로 저장하고, 날아갔다면 model 오브젝트로 직접 저장
try:
    if 'trainer' in locals() or 'trainer' in globals():
        trainer.save_model(output_dir)
        tokenizer.save_model(output_dir)
    else:
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
    print("🎉 [성공] 최종 학습 완료 모델 및 토크나이저가 안전하게 저장되었습니다!")
    print(f"📁 저장 경로: {os.path.abspath(output_dir)}")
except NameError:
    # 혹시나 model 변수까지 리셋되었을 경우를 위한 안전장치
    print("⚠️ 코랩 세션이 완전히 초기화된 것 같습니다. '학습 시작' 셀(trainer.train())을 다시 한번 실행한 뒤 이 셀을 돌려주세요!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 [성공] 최종 학습 완료 모델 및 토크나이저가 안전하게 저장되었습니다!
📁 저장 경로: /content/final_kobert_allergen_model
